# 22 — Learning values, learning policies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/022-two-solution-families.ipynb)

There are two ways to improve a behaviour: learn how good each action is and then take the best, or adjust the action probabilities directly from the reward. This notebook runs Q-learning on a two-state toy and REINFORCE on a two-armed bandit, then shows what happens when the step size is too large.

## 1. The numbers: a two-state toy and an empty Q table

Two states, two actions, and a table with one entry per pair. Right from state 0 reaches state 1 and pays 1; everything else is a smaller, quieter payoff. The table starts at zero, and one update already fills in a number.

In [ ]:
import numpy as np                              # the array library, preinstalled on Colab

def step(s, a):                                 # the toy environment: (state, action) -> (next state, reward)
    if s == 0 and a == 1:                       # from state 0, move right
        return 1, 1.0                           # reach state 1, collect 1
    if s == 0 and a == 0:                       # from state 0, stay left
        return 0, 0.0                           # stay put, collect nothing
    if s == 1 and a == 1:                       # from state 1, move right
        return 1, 0.2                           # stay in state 1, small trickle of reward
    return 0, 0.5                               # from state 1, move left: back to 0 with 0.5

Q = np.zeros((2, 2))                            # Q[state, action], all zero: nothing known yet
alpha = 0.5                                     # step size: how far Q moves toward the target
print("Q before any update:")                   # an empty table
print(Q)                                        # 2x2 zeros
print("Q[0, left] =", Q[0, 0])                  # an entry before it has ever been updated
target = 1.0 + 0.9 * Q[1].max()                 # reward plus discounted best value in the next state
Q[0, 1] = Q[0, 1] + alpha * (target - Q[0, 1])  # one Q-learning update for (state 0, right)
print("Q[0, right] after one update:", Q[0, 1])       # 0.5: halfway to the target of 1.0

**Line by line**

- `import numpy as np` — loads NumPy, the array library Colab already has.
- `def step(s, a)` — the whole environment in four lines.
- `if s == 0 and a == 1` — the good move: state 0 to state 1 for a reward of 1.
- `Q = np.zeros((2, 2))` — the table being learned, rows are states, columns actions.
- `alpha = 0.5` — the step size, the fraction of the gap to close each update.
- `target = 1.0 + 0.9 * Q[1].max()` — the bootstrap: this reward plus the discounted best next value.
- `Q[0, 1] = Q[0, 1] + alpha * (target - Q[0, 1])` — move `alpha` of the way toward the target.

## 2. Step by step: the update, then the whole run

Each step samples an action, observes a reward and a next state, then corrects one table entry. The correction is always reward plus discounted best-next-value minus what the entry currently claims. Wrapped in a function, the same loop can be rerun with a different step size later.

In [ ]:
np.random.seed(0)                               # fix the randomness for the printed trace
Q = np.zeros((2, 2))                            # reset the table
s = 0                                           # start the episode in state 0
for i in range(6):                              # six updates, printed one per line
    a = int(np.random.randint(2))               # a random action, so the trace shows both branches
    nxt, rew = step(s, a)                       # ask the environment what happens
    target = rew + 0.9 * Q[nxt].max()           # reward plus discounted best value of the next state
    Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])   # the Q-learning correction
    print(f"({s},{a}) -> {nxt}  r={rew}  target={target:.2f}  Q={Q[s, a]:.2f}")   # trace one update
    s = nxt                                     # continue the episode from where we landed

def train(alpha=0.5, episodes=300, seed=0, noise=0.0):   # the same loop, packaged for reuse
    rng = np.random.default_rng(seed)           # fresh randomness for each call
    Q = np.zeros((2, 2))                        # a fresh table
    hist = []                                   # the table after every episode
    for ep in range(episodes):                  # one episode is twenty steps
        s = 0                                   # every episode starts in state 0
        for _ in range(20):                     # short episodes keep this cheap
            if rng.random() < 0.2:              # epsilon = 0.2: sometimes explore
                a = int(rng.integers(2))        # a random action
            else:                               # otherwise
                a = int(np.argmax(Q[s]))        # the best action known for this state
            nxt, rew = step(s, a)               # the environment answers
            rew = rew + rng.normal(0, noise)    # optionally a noisy reward measurement
            target = rew + 0.9 * Q[nxt].max()   # the bootstrap target
            Q[s, a] = Q[s, a] + alpha * (target - Q[s, a])   # the update
            s = nxt                             # continue
        hist.append(Q.copy())                   # store the table, not a reference to it
    return np.array(hist)                       # shape (episodes, 2, 2)

**Line by line**

- `np.random.seed(0)` — makes the printed trace reproducible.
- `for i in range(6)` — six transitions, enough to see the pattern.
- `a = int(np.random.randint(2))` — a random action, standing in for exploration.
- `nxt, rew = step(s, a)` — the environment returns the next state and reward.
- `target = rew + 0.9 * Q[nxt].max()` — the number the entry is nudged toward.
- `Q[s, a] = ...` — only the visited pair changes; everything else is untouched.
- `def train(...)` — the same loop as a function, so sections 3, 4 and 6 can rerun it.
- `hist.append(Q.copy())` — `copy` matters, or every stored row would be the final table.

## 3. Plot: Q values finding their level

Four entries, four curves. The two greedy entries climb towards about 7.6 and 7.4, and the two dud entries settle lower; nothing tells the learner which is which, it comes out of the rewards. The ordering of the curves is the policy.

In [ ]:
import matplotlib.pyplot as plt                # plotting library, preinstalled on Colab

hist = train(alpha=0.5, episodes=300)           # the run from section 2, no noise
plt.figure(figsize=(6, 3))                      # a wide, short figure
plt.plot(hist[:, 0, 0], color="tab:red", label="Q(0, left)")       # a dud action
plt.plot(hist[:, 0, 1], color="tab:blue", label="Q(0, right)")     # the good action in state 0
plt.plot(hist[:, 1, 0], color="tab:green", label="Q(1, left)")     # the good action in state 1
plt.plot(hist[:, 1, 1], color="tab:orange", label="Q(1, right)")   # a worse action in state 1
plt.axhline(7.63, color="grey", linestyle="--", linewidth=0.8)     # true Q(0, right)
plt.axhline(7.37, color="grey", linestyle="--", linewidth=0.8)     # true Q(1, left)
plt.xlabel("episode")                           # x-axis label
plt.ylabel("Q(state, action)")                  # y-axis label
plt.legend(loc="lower right")                   # name the four curves
plt.title("Q(0,right) and Q(1,left) lead: the policy is read off, not stored")   # takeaway
plt.show()                                      # render the figure

**Line by line**

- `import matplotlib.pyplot as plt` — the plotting library, aliased to `plt`.
- `hist = train(alpha=0.5, episodes=300)` — one run of the learner, keeping its whole history.
- `hist[:, 0, 1]` — every episode's entry for state 0, action right.
- `plt.axhline(7.63)` — the value the Bellman equations give for that entry.
- `plt.plot(...)` — four curves on shared axes, so their order is the comparison.

## 4. Interactive: drag the step size

Alpha decides how far each update moves. Small values crawl but average out the noise; large values arrive fast and then twitch. Drag it and watch how quickly the two leading curves separate, with the final estimate printed in the title.

In [ ]:
from ipywidgets import interact               # sliders that rerun a function

def show(alpha=0.5):                            # alpha comes from the slider
    h = train(alpha=alpha, episodes=300)        # rerun the whole learner
    plt.figure(figsize=(6, 3))                  # fresh figure for each drag
    plt.plot(h[:, 0, 0], color="tab:red", label="Q(0, left)")       # dud action
    plt.plot(h[:, 0, 1], color="tab:blue", label="Q(0, right)")     # good action in state 0
    plt.plot(h[:, 1, 0], color="tab:green", label="Q(1, left)")     # good action in state 1
    plt.plot(h[:, 1, 1], color="tab:orange", label="Q(1, right)")   # worse action in state 1
    plt.axhline(7.63, color="grey", linestyle="--", linewidth=0.8)  # the true value
    plt.ylim(0, 9)                              # fixed axis, so drags compare fairly
    plt.xlabel("episode")                       # x-axis label
    plt.ylabel("Q(state, action)")              # y-axis label
    plt.legend(loc="lower right")               # name the curves
    plt.title(f"alpha={alpha:.2f} -> Q(0,right)={h[-1, 0, 1]:.2f}")   # live readout
    plt.show()                                  # draw it

interact(show, alpha=(0.02, 1.2, 0.02))         # the slider: range and step

**Line by line**

- `from ipywidgets import interact` — Colab ships the widgets library.
- `def show(alpha=0.5)` — the function that redraws one frame.
- `h = train(alpha=alpha, episodes=300)` — the only thing that changes between drags.
- `plt.ylim(0, 9)` — a fixed window, otherwise a jumpy run rescales the axis and hides itself.
- `plt.title(f"...")` — the readout of the final estimate.
- `interact(show, alpha=(0.02, 1.2, 0.02))` — attach the slider and its step size.

## 5. Policy without values: REINFORCE on a bandit

The second family of methods never builds a table. A policy holds one probability per action, and after each pull the probability of the action taken moves in proportion to the reward. Do it enough times and the good arm's probability climbs to 1.

In [ ]:
means2 = np.array([0.2, 0.9])                  # the two arms' hidden averages
theta = np.zeros(2)                             # preference for each action; equal, so 50/50
lr = 0.05                                       # policy step size
rng = np.random.default_rng(3)                  # seeded, so the run is reproducible
p_best = []                                     # probability of the good arm after each pull
for t in range(600):                            # six hundred pulls
    p = np.exp(theta) / np.exp(theta).sum()     # softmax: preferences turned into probabilities
    a = int(rng.choice(2, p=p))                 # sample the action from the policy itself
    rew = rng.normal(means2[a], 0.3)            # play it, get a noisy reward
    grad = -p.copy()                            # d log pi / d theta for the action not taken
    grad[a] += 1.0                              # plus one for the action actually taken
    theta = theta + lr * rew * grad             # reward times gradient: good pulls raise their odds
    p_new = np.exp(theta) / np.exp(theta).sum()  # the updated policy
    p_best.append(p_new[1])                     # track the good arm

print("policy at the start: 0.50 / 0.50")        # symmetric preferences
print(f"policy at the end: arm 0 = {1 - p_best[-1]:.3f}, arm 1 = {p_best[-1]:.3f}")   # the good arm dominates

plt.figure(figsize=(6, 3))                      # the learning curve
plt.plot(p_best, color="tab:purple")            # probability of arm 1 over time
plt.axhline(0.5, color="grey", linestyle="--", linewidth=0.8)   # the starting policy
plt.ylim(0, 1.02)                               # probabilities live on 0..1
plt.xlabel("pull number")                       # x-axis label
plt.ylabel("probability of choosing arm 1")     # y-axis label
plt.title("no values anywhere: the probability itself is what learns")   # the takeaway
plt.show()                                      # render the figure

**Line by line**

- `means2 = ...` — two arms, one clearly better, and no state at all.
- `theta = np.zeros(2)` — the policy's two preferences, equal at the start.
- `p = np.exp(theta) / np.exp(theta).sum()` — softmax, so preferences become probabilities.
- `a = int(rng.choice(2, p=p))` — the action is drawn, not chosen greedily.
- `grad = -p.copy()` then `grad[a] += 1.0` — the gradient of the log-probability of the action taken.
- `theta = theta + lr * rew * grad` — nudge each preference by reward times gradient.
- `p_best.append(p_new[1])` — one number per pull, the whole learning curve.
- `plt.plot(p_best)` — the curve rising from 0.5 to 1.

## 6. When it breaks: a step size that never averages

With alpha at 0.05 each update keeps 95% of the old estimate, so the noise cancels out over many updates. With alpha at 1.0 the old estimate is thrown away on every step and Q simply equals the last target, so it jitters with the reward instead of settling. Push alpha past 1 and the corrections overshoot in the other direction.

In [ ]:
smooth = train(alpha=0.05, episodes=300, seed=5, noise=0.4)   # patient step size
jumpy = train(alpha=1.0, episodes=300, seed=5, noise=0.4)    # alpha = 1.0: overwrite, never average

plt.figure(figsize=(6, 3))                      # compare the two traces
plt.plot(smooth[:, 0, 1], color="tab:green", label="alpha = 0.05")   # a settling estimate
plt.plot(jumpy[:, 0, 1], color="tab:red", label="alpha = 1.0")       # a jittering one
plt.axhline(7.63, color="grey", linestyle="--", linewidth=0.8)       # the true value
plt.xlabel("episode")                           # x-axis label
plt.ylabel("Q(0, right)")                       # y-axis label
plt.legend(loc="lower right")                   # name the two curves
plt.title("a large step size chases the noise instead of averaging it")   # the takeaway
plt.show()                                      # render

print("spread of the last 100 estimates, alpha=0.05:", round(np.ptp(smooth[-100:, 0, 1]), 2))   # tiny
print("spread of the last 100 estimates, alpha=1.00:", round(np.ptp(jumpy[-100:, 0, 1]), 2))    # large
print("final estimates:", round(smooth[-1, 0, 1], 2), "vs", round(jumpy[-1, 0, 1], 2))         # compare

**Line by line**

- `train(alpha=0.05, ..., noise=0.4)` — the same learner, now measuring noisy rewards.
- `train(alpha=1.0, ...)` — alpha of one means Q is replaced by the target each time.
- `plt.axhline(7.63)` — the value both runs are aiming at.
- `np.ptp(...)` — the range of the last hundred estimates: spread is the symptom.
- `print(...)` — the final estimates, which differ even though the seed is shared.

## Your turn

In section 6, predict the spread for alpha = 0.5 before you run it, then check. Then change `lr` in section 5 to 0.5 and explain why the policy probability jumps around instead of settling — is it the same disease as the alpha that is too large?